# CO oxidation on Pt: steady state and degree of rate control

A Langmuir-Hinshelwood mechanism written in the `discopt.mkm` operator syntax. We solve the steady-state coverages, compute the turnover frequency, and get the **Campbell degree of rate control** (DRC) and **thermodynamic rate control** (TRC) by automatic differentiation through the steady state.

## Build the mechanism

Species carry constant thermodynamics (`H`, `S`). Reactions use `reactants >> products`. Forward rates are Arrhenius; reverse rates are derived from the species thermodynamics so the model is thermodynamically consistent. Units here are eV / eV·K⁻¹ (`R` defaults to k_B).

In [1]:
import discopt.mkm as mk
from discopt.mkm.analysis import degree_of_rate_control, thermo_rate_control

m = mk.Model('co_oxidation', T=500.0)
s   = m.site('Pt', density=1.0)
CO  = m.gas('CO',  H=0.0,  S=0.0020)
O2  = m.gas('O2',  H=0.0,  S=0.0021)
CO2 = m.gas('CO2', H=-3.0, S=0.0023)
COs = m.adsorbate('CO*', site=s, H=-0.8, S=0.0005)
Os  = m.adsorbate('O*',  site=s, H=-0.3, S=0.0005)

m.step(CO + s >> COs,           A=1e4, Ea=0.0, name='CO adsorption')
m.step(O2 + 2*s >> 2*Os,        A=1e4, Ea=0.0, name='O2 dissociation')
m.step(COs + Os >> CO2 + 2*s,   A=1e8, Ea=0.7, name='surface reaction')
m

#,reaction,kinetics,type
1,CO + Pt ⇌ CO∗,"A=10000, Ea=0",reversible
2,O2 + 2 Pt ⇌ 2 O∗,"A=10000, Ea=0",reversible
3,CO∗ + O∗ ⇌ CO2 + 2 Pt,"A=1e+08, Ea=0.7",reversible


## Solve the steady state

The differential reactor holds the gas at fixed partial pressures.

In [2]:
reactor = mk.DifferentialReactor({CO: 1.0, O2: 0.5, CO2: 0.0})
sol = mk.solve_steady_state(m, reactor)
print('status:', sol.status)
for a in m.adsorbates:
    print(f'  theta[{a.name}] = {sol.coverage(a):.4f}')
print(f'  theta_free[Pt] = {sol.free_coverage(s):.4f}')
print(f'  TOF (CO2)      = {sol.production_rate(CO2):.4e}')

status: optimal
  theta[CO*] = 0.5852
  theta[O*] = 0.2314
  theta_free[Pt] = 0.1835
  TOF (CO2)      = 1.1902e+00


## Degree of rate control

`X_RC,i = (k_i/r) dr/dk_i` holding the other rate constants and all equilibrium constants fixed. By Campbell's theorem the steady-state DRC sums to 1.

We specify the species that we use to determine the rate for.

In [3]:
X = degree_of_rate_control(sol, species=CO2)
for rxn, x in X.items():
    print(f'  {rxn.name:18s}: {x:+.4f}')
print(f'  sum = {sum(X.values()):+.4f}')

  CO adsorption     : -0.0001
  O2 dissociation   : +0.0010
  surface reaction  : +0.9992
  sum = +1.0000


## Thermodynamic rate control

Sensitivity of the rate to each species' free energy.

In [4]:
for sp, x in thermo_rate_control(sol, species=CO2).items():
    print(f'  {sp.name:5s}: {x:+.4f}')

  CO   : +0.1712
  O2   : -0.2682
  CO2  : -0.0000
  Pt   : -0.3652
  CO*  : -0.1712
  O*   : +0.5364
